In [4]:
!pip install textstat==0.7.7

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 16.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 57.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.4/939.4 kB 55.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [13]:
# Project Review Status Classification - Data Preparation
# ======================================================

# This notebook handles the data preparation for the project review status classification model
# It includes text preprocessing, feature extraction from JSON fields, date feature engineering,
# and handling missing values.

# Table of Contents:
# 1. Import Libraries and Load Data
# 2. Exploratory Data Analysis
# 3. Text Preprocessing
# 4. JSON Field Processing
# 5. Date Feature Engineering
# 6. Handle Missing Values
# 7. Feature Consolidation
# 8. Export Processed Dataset

# 1. Import Libraries and Load Data
# ---------------------------------

import pandas as pd
import numpy as np
import json
import re
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import string
import textstat  # For readability metrics

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('vader_lexicon')

# Set the display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Load the dataset
# Remove the leading slash and use the correct file path
try:
    df = pd.read_csv("synthetic_projects_data.csv")
    print(f"Dataset Shape: {df.shape}")
    df.info()
except FileNotFoundError:
    print("Error: File 'synthetic_projects_data.csv' not found in the current directory.")
    print("Please ensure the file exists and the path is correct.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
Dataset Shape: (10000, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   project_id                      10000 non-null  int64  
 1   project_type_id                 10000 non-null  object 
 2   title                           10000 non-null  object 
 3   description                     10000 non-null  object 
 4   start_date            

In [16]:
# 2. Exploratory Data Analysis
# ----------------------------

# Display class distribution
print("\nReview Status Distribution:")
status_counts = df['review_status'].value_counts()
print(status_counts)

# Plot the distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='review_status', data=df)
plt.title('Distribution of Review Status')
plt.ylabel('Count')
plt.xlabel('Review Status')
plt.savefig('review_status_distribution.png')
plt.close()

# Project type distribution
print("\nProject Type Distribution:")
print(df['project_type_id'].value_counts())

# Check for missing values
print("\nMissing Values by Column:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])


Review Status Distribution:
review_status
Approved        4048
Rejected        2994
Needs Review    2958
Name: count, dtype: int64

Project Type Distribution:
project_type_id
Documentary    2022
TV             2019
Film           2002
Short Film     1985
Theater        1972
Name: count, dtype: int64

Missing Values by Column:
Series([], dtype: int64)


In [22]:
# Add the missing NLTK download for omw-1.4 resource
nltk.download('omw-1.4')

# 3. Text Preprocessing
# ---------------------

def preprocess_text(text, remove_stopwords=True, stem=False, lemmatize=True):
    """
    Preprocess text data: lowercase, remove punctuation, stopwords, and apply stemming/lemmatization
    
    Args:
        text (str): The text to preprocess
        remove_stopwords (bool): Whether to remove stopwords
        stem (bool): Whether to apply stemming
        lemmatize (bool): Whether to apply lemmatization
    
    Returns:
        str: Preprocessed text
    """
    if pd.isna(text) or text == '':
        return ''
    
    # Convert to string if not already
    if not isinstance(text, str):
        text = str(text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words]
    
    # Apply stemming
    if stem:
        stemmer = PorterStemmer()
        tokens = [stemmer.stem(word) for word in tokens]
    
    # Apply lemmatization
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Join tokens back into text
    preprocessed_text = ' '.join(tokens)
    
    return preprocessed_text

# Initialize sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Apply text preprocessing to text fields
print("Applying text preprocessing...")

# Create new columns for preprocessed text
df['title_processed'] = df['title'].apply(preprocess_text)
df['description_processed'] = df['description'].apply(preprocess_text)
df['synopsis_processed'] = df['synopsis'].apply(preprocess_text)

# Generate text features
def extract_text_features(row):
    """Extract features from text fields"""
    # Text length features
    title_length = len(str(row['title'])) if pd.notna(row['title']) else 0
    desc_length = len(str(row['description'])) if pd.notna(row['description']) else 0
    synopsis_length = len(str(row['synopsis'])) if pd.notna(row['synopsis']) else 0
    
    # Word count features
    title_word_count = len(str(row['title']).split()) if pd.notna(row['title']) else 0
    desc_word_count = len(str(row['description']).split()) if pd.notna(row['description']) else 0
    synopsis_word_count = len(str(row['synopsis']).split()) if pd.notna(row['synopsis']) else 0
    
    # Sentiment features (using VADER)
    title_sentiment = sia.polarity_scores(str(row['title']))['compound'] if pd.notna(row['title']) else 0
    desc_sentiment = sia.polarity_scores(str(row['description']))['compound'] if pd.notna(row['description']) else 0
    synopsis_sentiment = sia.polarity_scores(str(row['synopsis']))['compound'] if pd.notna(row['synopsis']) else 0
    
    # Readability features (using textstat)
    desc_readability = textstat.flesch_reading_ease(str(row['description'])) if pd.notna(row['description']) else 0
    synopsis_readability = textstat.flesch_reading_ease(str(row['synopsis'])) if pd.notna(row['synopsis']) else 0
    
    return pd.Series({
        'title_length': title_length,
        'description_length': desc_length,
        'synopsis_length': synopsis_length,
        'title_word_count': title_word_count,
        'description_word_count': desc_word_count,
        'synopsis_word_count': synopsis_word_count,
        'title_sentiment': title_sentiment,
        'description_sentiment': desc_sentiment,
        'synopsis_sentiment': synopsis_sentiment,
        'description_readability': desc_readability,
        'synopsis_readability': synopsis_readability
    })

# Extract text features
print("Extracting text features...")
text_features = df.apply(extract_text_features, axis=1)
df = pd.concat([df, text_features], axis=1)

[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
Applying text preprocessing...
Extracting text features...


In [25]:
# 4. JSON Field Processing
# ------------------------

def process_production_locations(locations_str):
    """Extract features from production_locations JSON"""
    if pd.isna(locations_str):
        return pd.Series({
            'num_locations': 0,
            'has_major_city': False
        })
    
    try:
        # Parse JSON if it's a string
        if isinstance(locations_str, str):
            locations = json.loads(locations_str)
        else:
            locations = locations_str
            
        # Count locations
        num_locations = len(locations)
        
        # Check if major cities are included (can be expanded with more cities)
        major_cities = ['Colombo', 'Kandy', 'Galle', 'Jaffna']
        has_major_city = any(city in locations for city in major_cities)
        
        return pd.Series({
            'num_locations': num_locations,
            'has_major_city': has_major_city
        })
    except:
        return pd.Series({
            'num_locations': 0,
            'has_major_city': False
        })

def process_production_dates(dates_str):
    """Extract features from production_dates JSON"""
    if pd.isna(dates_str):
        return pd.Series({
            'rehearsal_to_shooting_days': 0,
            'production_duration_months': 0,
            'is_summer_production': False,
            'is_winter_production': False
        })
    
    try:
        # Parse JSON if it's a string
        if isinstance(dates_str, str):
            dates = json.loads(dates_str)
        else:
            dates = dates_str
            
        # Extract dates
        rehearsal_date = dates.get('rehearsal_date', None)
        shooting_start = dates.get('shooting_start', None)
        duration_str = dates.get('duration', '0 months')
        
        # Calculate days between rehearsal and shooting
        rehearsal_to_shooting_days = 0
        if rehearsal_date and shooting_start:
            rehearsal = datetime.strptime(rehearsal_date, '%Y-%m-%d')
            shooting = datetime.strptime(shooting_start, '%Y-%m-%d')
            rehearsal_to_shooting_days = (shooting - rehearsal).days
        
        # Extract production duration in months
        production_duration_months = 0
        if isinstance(duration_str, str):
            duration_match = re.search(r'(\d+)\s*months', duration_str)
            if duration_match:
                production_duration_months = int(duration_match.group(1))
        
        # Determine season (Northern Hemisphere seasons)
        is_summer_production = False
        is_winter_production = False
        if shooting_start:
            shooting_month = datetime.strptime(shooting_start, '%Y-%m-%d').month
            is_summer_production = 6 <= shooting_month <= 8  # June, July, August
            is_winter_production = shooting_month <= 2 or shooting_month == 12  # Dec, Jan, Feb
        
        return pd.Series({
            'rehearsal_to_shooting_days': rehearsal_to_shooting_days,
            'production_duration_months': production_duration_months,
            'is_summer_production': is_summer_production,
            'is_winter_production': is_winter_production
        })
    except:
        return pd.Series({
            'rehearsal_to_shooting_days': 0,
            'production_duration_months': 0,
            'is_summer_production': False,
            'is_winter_production': False
        })

def extract_compensation_value(comp_str):
    """Extract numeric value from compensation string"""
    if pd.isna(comp_str) or comp_str == 'Negotiable':
        return np.nan
    
    try:
        # Look for patterns like "36K LKR"
        match = re.search(r'(\d+)K', comp_str)
        if match:
            return int(match.group(1)) * 1000
        
        # If it's just a number
        match = re.search(r'(\d+)', comp_str)
        if match:
            return int(match.group(1))
    except:
        pass
    
    return np.nan

def process_crew_roles(crew_str):
    """Extract features from crew_roles JSON"""
    if pd.isna(crew_str):
        return pd.Series({
            'num_crew_roles': 0,
            'has_director': False,
            'has_assistant_director': False,
            'avg_crew_compensation': 0,
            'max_crew_compensation': 0,
            'pct_negotiable_crew': 0
        })
    
    try:
        # Parse JSON if it's a string
        if isinstance(crew_str, str):
            crew = json.loads(crew_str)
        else:
            crew = crew_str
            
        # Count roles
        num_crew_roles = len(crew)
        
        # Check for key roles
        has_director = any(role.get('role_name', '').lower() == 'director' for role in crew)
        has_assistant_director = any('assistant director' in role.get('role_name', '').lower() for role in crew)
        
        # Analyze compensation
        compensations = []
        negotiable_count = 0
        
        for role in crew:
            comp = role.get('compensation', '')
            if comp == 'Negotiable':
                negotiable_count += 1
            else:
                comp_value = extract_compensation_value(comp)
                if not pd.isna(comp_value):
                    compensations.append(comp_value)
        
        # Calculate compensation metrics
        avg_comp = np.mean(compensations) if compensations else 0
        max_comp = max(compensations) if compensations else 0
        pct_negotiable = (negotiable_count / num_crew_roles * 100) if num_crew_roles > 0 else 0
        
        return pd.Series({
            'num_crew_roles': num_crew_roles,
            'has_director': has_director,
            'has_assistant_director': has_assistant_director,
            'avg_crew_compensation': avg_comp,
            'max_crew_compensation': max_comp,
            'pct_negotiable_crew': pct_negotiable
        })
    except:
        return pd.Series({
            'num_crew_roles': 0,
            'has_director': False,
            'has_assistant_director': False,
            'avg_crew_compensation': 0,
            'max_crew_compensation': 0,
            'pct_negotiable_crew': 0
        })

def process_casting_roles(casting_str):
    """Extract features from casting_roles JSON"""
    if pd.isna(casting_str):
        return pd.Series({
            'num_casting_roles': 0,
            'num_male_roles': 0,
            'num_female_roles': 0,
            'num_nonbinary_roles': 0,
            'gender_diversity_score': 0,
            'avg_casting_compensation': 0,
            'max_casting_compensation': 0,
            'pct_negotiable_casting': 0,
            'has_child_actors': False,
            'age_range_diversity': 0,
            'casting_description_sentiment': 0
        })
    
    try:
        # Parse JSON if it's a string
        if isinstance(casting_str, str):
            casting = json.loads(casting_str)
        else:
            casting = casting_str
            
        # Count roles
        num_casting_roles = len(casting)
        
        # Gender analysis
        num_male_roles = sum(1 for role in casting if role.get('gender', '').lower() == 'male')
        num_female_roles = sum(1 for role in casting if role.get('gender', '').lower() == 'female')
        num_nonbinary_roles = sum(1 for role in casting if role.get('gender', '').lower() == 'non-binary')
        
        # Gender diversity score (higher = more diverse)
        total_roles = num_male_roles + num_female_roles + num_nonbinary_roles
        if total_roles > 0:
            gender_diversity_score = 1 - ((num_male_roles / total_roles)**2 + 
                                        (num_female_roles / total_roles)**2 + 
                                        (num_nonbinary_roles / total_roles)**2)
        else:
            gender_diversity_score = 0
        
        # Compensation analysis
        compensations = []
        negotiable_count = 0
        
        for role in casting:
            comp = role.get('compensation', '')
            if comp == 'Negotiable':
                negotiable_count += 1
            else:
                comp_value = extract_compensation_value(comp)
                if not pd.isna(comp_value):
                    compensations.append(comp_value)
        
        # Calculate compensation metrics
        avg_comp = np.mean(compensations) if compensations else 0
        max_comp = max(compensations) if compensations else 0
        pct_negotiable = (negotiable_count / num_casting_roles * 100) if num_casting_roles > 0 else 0
        
        # Age range analysis
        age_ranges = set()
        has_child_actors = False
        
        for role in casting:
            age_range = role.get('age_range', '')
            if age_range:
                age_ranges.add(age_range)
                # Check for child actors (under 18)
                if '6-12' in age_range or '13-17' in age_range:
                    has_child_actors = True
        
        # Age diversity score (more unique age ranges = higher diversity)
        age_range_diversity = len(age_ranges)
        
        # Sentiment analysis on role descriptions
        description_sentiments = []
        for role in casting:
            desc = role.get('description', '')
            if desc:
                sentiment = sia.polarity_scores(desc)['compound']
                description_sentiments.append(sentiment)
        
        # Average sentiment
        avg_description_sentiment = np.mean(description_sentiments) if description_sentiments else 0
        
        return pd.Series({
            'num_casting_roles': num_casting_roles,
            'num_male_roles': num_male_roles,
            'num_female_roles': num_female_roles,
            'num_nonbinary_roles': num_nonbinary_roles,
            'gender_diversity_score': gender_diversity_score,
            'avg_casting_compensation': avg_comp,
            'max_casting_compensation': max_comp,
            'pct_negotiable_casting': pct_negotiable,
            'has_child_actors': has_child_actors,
            'age_range_diversity': len(age_ranges),
            'casting_description_sentiment': avg_description_sentiment
        })
    except:
        return pd.Series({
            'num_casting_roles': 0,
            'num_male_roles': 0,
            'num_female_roles': 0,
            'num_nonbinary_roles': 0,
            'gender_diversity_score': 0,
            'avg_casting_compensation': 0,
            'max_casting_compensation': 0,
            'pct_negotiable_casting': 0,
            'has_child_actors': False,
            'age_range_diversity': 0,
            'casting_description_sentiment': 0
        })

# Process JSON fields
print("Processing JSON fields...")
location_features = df['production_locations'].apply(process_production_locations)
date_features = df['production_dates'].apply(process_production_dates)
crew_features = df['crew_roles'].apply(process_crew_roles)
casting_features = df['casting_roles'].apply(process_casting_roles)

# Add the extracted features to the dataframe
df = pd.concat([df, location_features, date_features, crew_features, casting_features], axis=1)


Processing JSON fields...


In [28]:
# 5. Date Feature Engineering
# ---------------------------

def process_dates(row):
    """Extract features from date fields"""
    try:
        # Convert date strings to datetime objects
        start_date = pd.to_datetime(row['start_date']) if pd.notna(row['start_date']) else None
        end_date = pd.to_datetime(row['end_date']) if pd.notna(row['end_date']) else None
        
        # Initialize features
        project_duration_days = np.nan
        project_year = np.nan
        project_month = np.nan
        is_holiday_season = False
        
        if start_date and end_date:
            # Calculate project duration
            project_duration_days = (end_date - start_date).days
            
            # Extract year and month
            project_year = start_date.year
            project_month = start_date.month
            
            # Check if project starts during holiday season (Nov-Jan)
            is_holiday_season = 11 <= start_date.month <= 12 or start_date.month == 1
        
        return pd.Series({
            'project_duration_days': project_duration_days,
            'project_year': project_year,
            'project_month': project_month,
            'is_holiday_season': is_holiday_season
        })
    except:
        return pd.Series({
            'project_duration_days': np.nan,
            'project_year': np.nan,
            'project_month': np.nan,
            'is_holiday_season': False
        })

# Process date fields
print("Engineering date features...")
date_features = df.apply(process_dates, axis=1)
df = pd.concat([df, date_features], axis=1)

Engineering date features...


In [31]:
# 6. Handle Missing Values
# ------------------------

# Check for missing values after feature engineering
print("\nMissing values after feature engineering:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

# Define numerical and categorical columns
numerical_cols = [
    'synopsis_plagiarism_similarity', 'title_length', 'description_length', 'synopsis_length',
    'title_word_count', 'description_word_count', 'synopsis_word_count',
    'title_sentiment', 'description_sentiment', 'synopsis_sentiment',
    'description_readability', 'synopsis_readability', 
    'num_locations', 'rehearsal_to_shooting_days', 'production_duration_months',
    'num_crew_roles', 'avg_crew_compensation', 'max_crew_compensation', 'pct_negotiable_crew',
    'num_casting_roles', 'num_male_roles', 'num_female_roles', 'num_nonbinary_roles',
    'gender_diversity_score', 'avg_casting_compensation', 'max_casting_compensation',
    'pct_negotiable_casting', 'age_range_diversity', 'casting_description_sentiment',
    'project_duration_days', 'project_year', 'project_month'
]

categorical_cols = [
    'project_type_id', 'package_type_of_user', 'has_major_city', 
    'is_summer_production', 'is_winter_production', 'has_director',
    'has_assistant_director', 'has_child_actors', 'is_holiday_season'
]

# Impute missing values
from sklearn.impute import SimpleImputer

# For numerical columns
print("Imputing missing values for numerical columns...")
num_imputer = SimpleImputer(strategy='median')
df[numerical_cols] = num_imputer.fit_transform(df[numerical_cols])

# For categorical columns
print("Imputing missing values for categorical columns...")
cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])


Missing values after feature engineering:
Series([], dtype: int64)
Imputing missing values for numerical columns...
Imputing missing values for categorical columns...


In [34]:
# 7. Feature Consolidation
# ------------------------

# Create a list of all features we want to keep for the model
model_features = numerical_cols + categorical_cols

# Add the target column
model_features.append('review_status')

# Create the final dataframe with selected features
df_final = df[model_features].copy()

# Check the final dataframe
print("\nFinal dataframe shape:", df_final.shape)
print("\nFinal features:", df_final.columns.tolist())

# Quick look at the data
print("\nSample of processed data:")
print(df_final.head())

# Generate descriptive statistics
print("\nDescriptive statistics:")
print(df_final.describe())



Final dataframe shape: (10000, 42)

Final features: ['synopsis_plagiarism_similarity', 'title_length', 'description_length', 'synopsis_length', 'title_word_count', 'description_word_count', 'synopsis_word_count', 'title_sentiment', 'description_sentiment', 'synopsis_sentiment', 'description_readability', 'synopsis_readability', 'num_locations', 'rehearsal_to_shooting_days', 'production_duration_months', 'num_crew_roles', 'avg_crew_compensation', 'max_crew_compensation', 'pct_negotiable_crew', 'num_casting_roles', 'num_male_roles', 'num_female_roles', 'num_nonbinary_roles', 'gender_diversity_score', 'avg_casting_compensation', 'max_casting_compensation', 'pct_negotiable_casting', 'age_range_diversity', 'casting_description_sentiment', 'project_duration_days', 'project_year', 'project_month', 'project_type_id', 'package_type_of_user', 'has_major_city', 'is_summer_production', 'is_winter_production', 'has_director', 'has_assistant_director', 'has_child_actors', 'is_holiday_season', 'revi

In [37]:
# 8. Export Processed Dataset
# ---------------------------

# Save the processed dataframe to a CSV file
df_final.to_csv('processed_project_data.csv', index=False)
print("\nProcessed data saved to 'processed_project_data.csv'")

# Generate a correlation heatmap for numerical features
plt.figure(figsize=(15, 12))
correlation = df_final[numerical_cols].corr()
sns.heatmap(correlation, annot=False, cmap='coolwarm')
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.close()

print("\nData preparation complete!")

# Additional helper to check the relationship between features and target
plt.figure(figsize=(15, 10))
sns.countplot(x='project_type_id', hue='review_status', data=df_final)
plt.title('Review Status by Project Type')
plt.xticks(rotation=45)
plt.savefig('review_status_by_project_type.png')
plt.close()

# If the dataset is not too large, we can also do a pairplot of key features
key_features = ['synopsis_plagiarism_similarity', 'num_locations', 'gender_diversity_score', 'review_status']
sns.pairplot(df_final[key_features], hue='review_status')
plt.savefig('pairplot_key_features.png')
plt.close()

print("Visualization files created.")


Processed data saved to 'processed_project_data.csv'

Data preparation complete!
Visualization files created.
